# REMS MODRDR: batch binning of many sols

Reads every MODRDR file (`RME_*RMD*.TAB`) in a folder, bins each sol's 1 Hz readings by **local true solar
time (LTST)**, and writes **one combined CSV** with a block of time-of-day bins per sol.

- Put the `RMD` `.TAB` files in one folder (subfolders are searched too). The `.LBL` files are not needed.
- Other REMS products in the same folder (`RNV`, `RTL`, `ADR`) are ignored.
- A file that fails to parse is reported with its location and the rest still run; the notebook then stops before
  writing the CSV unless `ALLOW_FAILURES = True`, so a failure cannot silently drop a sol.
- Optionally adds the rover's position for each sol (site, drive, latitude, longitude, elevation, roll, pitch,
  yaw) from the PDS `localized_interp.csv` table. Set `LOCATION_CSV = None` to skip.
- Same method as the single-sol notebook: `-999` is removed, bins with fewer than `MIN_SAMPLES` readings
  become empty, and every bin keeps a sample count (`n_...` columns).

**Run order:** edit the settings cell, then Run All.

In [1]:
from pathlib import Path

# ---- Settings -------------------------------------------------------------
INPUT_DIR = Path("raw_data/rems_tabs" )                # folder with the RMD .TAB files (subfolders are searched too)
BIN_MIN = 15                            # bin width in minutes (must divide 1440: 10, 15, 20, 30, 60...)
MIN_SAMPLES = 20                        # bins with fewer readings than this are set to NaN
STRICT = True                          # True = also drop readings whose confidence flag marks them bad
DROP_EMPTY_BINS = True                  # drop rows where air temp, ground temp and pressure are all empty
OUT_CSV = Path("processed_data/rems_binned_all_sols.csv")    # combined output file
LOCATION_CSV = Path("raw_data/localized_interp.csv")   # rover location table (PDS MSL places); None = skip the location join
LAT_COLUMN = "planetocentric_latitude"  # or "planetodetic_latitude" (differ by ~3 km at Gale)
ALLOW_FAILURES = False                  # False = stop before writing the CSV if any file fails to parse

In [2]:
import csv
import os
import sys
import traceback
from collections import Counter

import numpy as np
import pandas as pd

print("python", sys.version.split()[0], "| numpy", np.__version__, "| pandas", pd.__version__)

# Column order from MODRDR6.FMT (40 comma-separated columns).
COLS = [
    "TIMESTAMP", "LMST", "LTST",
    "HORIZONTAL_WIND_SPEED", "VERTICAL_WIND_SPEED", "WIND_DIRECTION", "WS_CONFIDENCE_LEVEL",
    "BRIGHTNESS_TEMP", "BRIGHTNESS_TEMP_LONG_TERM_UNCERTAINTY",
    "BRIGHTNESS_TEMP_SHORT_TERM_UNCERTAINTY", "GTS_CONFIDENCE_LEVEL",
    "BOOM1_LOCAL_AIR_TEMP", "ATS_BOOM1_CONFIDENCE_LEVEL",
    "BOOM2_LOCAL_AIR_TEMP", "ATS_BOOM2_CONFIDENCE_LEVEL",
    "AMBIENT_TEMP", "AMBIENT_TEMP_CONFIDENCE_LEVEL",
    "UV_A", "UV_B", "UV_C", "UV_ABC", "UV_D", "UV_E",
    "UV_A_UNCERTAINTY", "UV_B_UNCERTAINTY", "UV_C_UNCERTAINTY",
    "UV_ABC_UNCERTAINTY", "UV_D_UNCERTAINTY", "UV_E_UNCERTAINTY",
    "UVS_CONFIDENCE_LEVEL",
    "LOCAL_RELATIVE_HUMIDITY", "HS_TEMP", "LOCAL_RELATIVE_HUMIDITY_UNCERTAINTY",
    "VOLUME_MIXING_RATIO", "VOLUME_MIXING_RATIO_UNCERTAINTY", "HS_CONFIDENCE_LEVEL",
    "PS_CONFIGURATION", "PRESSURE", "PRESSURE_UNCERTAINTY", "PS_CONFIDENCE_LEVEL",
]
assert len(COLS) == 40

# output name, value column, flag column, position of the flag character that must not be "0"
# (strict mode only: air = last character, ground and pressure = second character)
TARGETS = [
    ("air_temp_k", "AMBIENT_TEMP", "AMBIENT_TEMP_CONFIDENCE_LEVEL", -1),
    ("ground_temp_k", "BRIGHTNESS_TEMP", "GTS_CONFIDENCE_LEVEL", 1),
    ("pressure_pa", "PRESSURE", "PS_CONFIDENCE_LEVEL", 1),
]
VALUE_COLS = [t[0] for t in TARGETS]
NEEDED = ["LMST", "LTST"] + [c for t in TARGETS for c in t[1:3]]

assert 1440 % BIN_MIN == 0, "BIN_MIN must divide 1440 evenly"
NBINS = 1440 // BIN_MIN

python 3.14.7 | numpy 2.5.2 | pandas 3.0.5


## 1. Find the files

In [3]:
if not os.path.isdir(INPUT_DIR):
    raise FileNotFoundError(f"INPUT_DIR not found: {os.path.abspath(INPUT_DIR)}")

files, ignored = [], 0
for root, _, names in os.walk(INPUT_DIR):
    for name in names:
        u = name.upper()
        if not u.endswith(".TAB"):
            continue
        if u.startswith("RME_") and "RMD" in u:
            files.append(os.path.join(root, name))
        else:
            ignored += 1
files.sort()

if not files:
    raise FileNotFoundError(f"No RME_*RMD*.TAB files found under {os.path.abspath(INPUT_DIR)}")

print(f"Found {len(files)} MODRDR files in {os.path.abspath(INPUT_DIR)}  ({ignored} other .TAB files ignored)")
print("first:", os.path.basename(files[0]))
print("last: ", os.path.basename(files[-1]))

Found 321 MODRDR files in c:\Users\aypen\Downloads\MarsDataPipeline\raw_data\rems_tabs  (1 other .TAB files ignored)
first: RME_397535244RMD00010000000_______PA.TAB
last:  RME_732836186RMD37780000000_______P2.TAB


## 2. Bin each sol

Each file is read, reduced to one block of bins, and released, so memory stays small even for hundreds of
sols. Parsing uses plain Python and numpy (no pandas string functions), matching the single-sol notebook.

In [4]:
def ltst_hours(s):
    # "00367 23:25:09" -> hours since local midnight (the sol number is ignored)
    t = str(s).split()[1]
    h, m, sec = t.split(":")
    return int(h) + int(m) / 60 + int(sec) / 3600


def flag_ok(f, pos):
    f = str(f).strip()
    try:
        return f[pos] != "0"
    except IndexError:
        return False


def process_file(path):
    with open(path, newline="") as fh:
        first = fh.readline()
    n_fields = len(next(csv.reader([first])))
    if n_fields != 40:
        raise ValueError(f"expected 40 columns, found {n_fields}")

    # Time and flag columns are read as text and types are not guessed chunk by chunk. With the default
    # low_memory=True, pandas' "mixed types" path raises IndexError when usecols is used (sols 26, 1343, ...).
    text_cols = ["LMST", "LTST"] + [t[2] for t in TARGETS]
    df = pd.read_csv(path, header=None, names=COLS, usecols=NEEDED, dtype={c: str for c in text_cols},
                     quotechar='"', skipinitialspace=True, low_memory=False)
    if len(df) == 0:
        raise ValueError("empty file")

    sol = int(Counter(str(s).strip()[:5] for s in df["LMST"]).most_common(1)[0][0])
    hours = np.array([ltst_hours(s) for s in df["LTST"]])
    bins = np.clip(np.floor(hours * 60 / BIN_MIN).astype(int), 0, NBINS - 1)

    out = pd.DataFrame({"sol": sol, "bin": np.arange(NBINS)})
    out["ltst_bin_start_hr"] = out["bin"] * BIN_MIN / 60
    out["ltst_bin_center_hr"] = out["ltst_bin_start_hr"] + BIN_MIN / 120

    info = {"sol": sol, "file": os.path.basename(path), "rows": len(df)}
    for name, col, flagcol, pos in TARGETS:
        v = np.array(pd.to_numeric(df[col], errors="coerce"), dtype=float)  # writable copy
        v[v <= -998] = np.nan
        valid = ~np.isnan(v)
        if STRICT:
            valid &= np.array([flag_ok(f, pos) for f in df[flagcol]])
        n = np.bincount(bins[valid], minlength=NBINS)
        total = np.bincount(bins[valid], weights=v[valid], minlength=NBINS)
        with np.errstate(invalid="ignore", divide="ignore"):
            mean = total / n
        mean[n < MIN_SAMPLES] = np.nan
        out[f"n_{name}"] = n
        out[name] = mean
        info[f"{name}_valid_pct"] = round(100 * valid.mean(), 1)
        info[f"{name}_bins"] = int(np.isfinite(mean).sum())
    return out, info


frames, records, failures = [], [], []
for i, path in enumerate(files, start=1):
    try:
        out, info = process_file(path)
    except Exception as e:
        at = traceback.extract_tb(e.__traceback__)[-1]
        why = f"{type(e).__name__}: {e} (at {os.path.basename(at.filename)}:{at.lineno} in {at.name})"
        failures.append((os.path.basename(path), why))
        print(f"[{i}/{len(files)}] FAILED {os.path.basename(path)}: {why}")
        continue
    frames.append(out)
    records.append(info)
    #print(f"[{i}/{len(files)}] sol {info['sol']}: {info['rows']:,} rows | bins filled "
          #f"air {info['air_temp_k_bins']}, ground {info['ground_temp_k_bins']}, "
          #f"pressure {info['pressure_pa_bins']} (of {NBINS})")

if not frames:
    raise RuntimeError("No files were processed successfully; see the FAILED lines above.")

print(f"Processed {len(frames)} files successfully, {len(failures)} failures.")
if failures and not ALLOW_FAILURES:
    raise RuntimeError(f"{len(failures)} files failed to parse (see the FAILED lines above). "
                       "Fix them, or set ALLOW_FAILURES = True to continue without them.")

Processed 321 files successfully, 0 failures.


## 3. Coverage per sol

One row per sol. `*_valid_pct` is the share of 1 Hz readings that were usable; `*_bins` is how many time-of-day
bins ended up with enough samples. Low ground-temperature or wind coverage in some sols is expected.

In [5]:
summary = pd.DataFrame(records).sort_values("sol").reset_index(drop=True)

dups = summary[summary["sol"].duplicated(keep=False)]
if len(dups):
    print("WARNING: these sols appear in more than one file (both are kept in the output):")
    print(dups[["sol", "file"]].to_string(index=False))

print(f"{len(summary)} sols processed, {len(failures)} files failed")
for name, why in failures:
    print("  FAILED", name, "-", why)

summary

 sol                                         file
2875 RME_652672884RMD28750000000_______P3 (1).TAB
2875     RME_652672884RMD28750000000_______P3.TAB
2878 RME_652939208RMD28780000000_______P3 (1).TAB
2878     RME_652939208RMD28780000000_______P3.TAB
321 sols processed, 0 files failed


,sol,file,rows,air_temp_k_valid_pct,air_temp_k_bins,ground_temp_k_valid_pct,ground_temp_k_bins,pressure_pa_valid_pct,pressure_pa_bins
0,1,RME_397535244RMD00010000000_______PA.TAB,750,64.0,1,68.0,2,60.0,2
1,9,RME_398245439RMD00090000000_______PA.TAB,5520,95.7,15,83.7,15,94.6,15
2,10,RME_398334213RMD00100000000_______PA.TAB,10200,100.0,27,84.1,27,100.0,27
3,11,RME_398422988RMD00110000000_______PA.TAB,12960,94.9,30,81.9,32,95.4,32
4,12,RME_398511762RMD00120000000_______PA.TAB,13200,100.0,30,88.2,30,100.0,30
...,...,...,...,...,...,...,...,...,...
316,3771,RME_732214765RMD37710000000_______P2.TAB,31129,100.0,45,95.2,45,96.2,44
317,3773,RME_732392314RMD37730000000_______P2.TAB,31068,100.0,45,95.4,45,96.8,45
318,3776,RME_732658637RMD37760000000_______P2.TAB,31129,100.0,45,95.6,45,98.5,45
319,3777,RME_732747412RMD37770000000_______P2.TAB,27738,100.0,42,94.8,42,100.0,42


## 4. Combine the sols

In [6]:
combined = pd.concat(frames, ignore_index=True)
combined = combined.sort_values(["sol", "bin"], kind="stable").reset_index(drop=True)
if DROP_EMPTY_BINS:
    combined = combined.dropna(subset=VALUE_COLS, how="all").reset_index(drop=True)
print(f"{len(combined):,} rows from {combined['sol'].nunique()} sols")

15,640 rows from 319 sols


## 5. Add the rover's position (optional)

`localized_interp.csv` has a row only when the rover's position changed (a drive or a pose update), so most
sols have no row of their own. For each sol, this uses the **last row at or before that sol**: the rover's
most recent known position. Only `ROVER` rows are used (the `SITE` rows are site-frame origins).

`loc_row_this_sol` is `True` when the table has a row for that exact sol, meaning the rover drove or its
pose was updated during the sol. Readings from those sols may come from more than one position, so you may
want to filter them out when training.

In [7]:
LOC_COLS = {
    "site": "rover_site",
    "drive": "rover_drive",
    LAT_COLUMN: "latitude_deg",
    "longitude": "longitude_deg_e",
    "elevation": "elevation_m",
    "roll": "roll_deg",
    "pitch": "pitch_deg",
    "yaw": "yaw_deg",
}

if LOCATION_CSV is None:
    print("LOCATION_CSV is None: skipping the location join")
else:
    if not os.path.isfile(LOCATION_CSV):
        raise FileNotFoundError(f"LOCATION_CSV not found: {os.path.abspath(LOCATION_CSV)}")
    loc = pd.read_csv(LOCATION_CSV)
    missing_cols = [c for c in ["frame", "sol"] + list(LOC_COLS) if c not in loc.columns]
    if missing_cols:
        raise ValueError(f"{LOCATION_CSV} is missing columns: {missing_cols}")

    loc = loc[loc["frame"] == "ROVER"]
    sols_with_row = set(loc["sol"])
    # one row per sol: the last one in file order (the file is ordered by time)
    loc = loc.drop_duplicates("sol", keep="last")[["sol"] + list(LOC_COLS)]
    loc = loc.rename(columns=LOC_COLS).sort_values("sol")

    n_before = len(combined)
    combined = pd.merge_asof(combined.sort_values("sol", kind="stable"), loc,
                             on="sol", direction="backward")
    combined["loc_row_this_sol"] = combined["sol"].isin(sols_with_row)
    assert len(combined) == n_before
    combined = combined.sort_values(["sol", "bin"], kind="stable").reset_index(drop=True)

    no_pos = combined["latitude_deg"].isna().sum()
    if no_pos:
        print(f"WARNING: {no_pos} rows have no position (sol earlier than the first location row)")
    last_loc_sol = int(loc["sol"].max())
    late = combined.loc[combined["sol"] > last_loc_sol, "sol"].nunique()
    if late:
        print(f"WARNING: {late} sols are later than the last location row (sol {last_loc_sol}); "
              "their position is just the last known one")
    print(f"Added position columns using {LAT_COLUMN}; "
          f"{int(combined['loc_row_this_sol'].groupby(combined['sol']).first().sum())} of "
          f"{combined['sol'].nunique()} sols have a location row of their own")

Added position columns using planetocentric_latitude; 101 of 319 sols have a location row of their own


## 6. Save the CSV

In [8]:
out_df = combined.round(2)
for c in ("latitude_deg", "longitude_deg_e"):
    if c in combined.columns:
        out_df[c] = combined[c].round(6)   # keep full precision for position

out_df.to_csv(OUT_CSV, index=False)
print(f"Wrote {OUT_CSV}: {len(out_df):,} rows, {out_df.shape[1]} columns, {out_df['sol'].nunique()} sols "
      f"({BIN_MIN}-min bins, min {MIN_SAMPLES} samples, strict={STRICT})")
out_df.head(10)

Wrote processed_data\rems_binned_all_sols.csv: 15,640 rows, 19 columns, 319 sols (15-min bins, min 20 samples, strict=True)


,sol,bin,ltst_bin_start_hr,ltst_bin_center_hr,n_air_temp_k,air_temp_k,n_ground_temp_k,ground_temp_k,n_pressure_pa,pressure_pa,rover_site,rover_drive,latitude_deg,longitude_deg_e,elevation_m,roll_deg,pitch_deg,yaw_deg,loc_row_this_sol
0,1,55,13.75,13.88,480,267.33,480,247.39,420,704.53,1,8,-4.589467,137.441633,-4501.04,-2.42,-3.62,111.52,False
1,1,56,14.00,14.12,0,NaN,30,234.04,30,702.84,1,8,-4.589467,137.441633,-4501.04,-2.42,-3.62,111.52,False
2,9,52,13.00,13.12,300,251.78,300,277.32,300,714.10,3,0,-4.589467,137.441633,-4501.04,-2.42,-3.62,111.52,False
3,9,54,13.50,13.62,319,257.28,319,281.71,259,709.09,3,0,-4.589467,137.441633,-4501.04,-2.42,-3.62,111.52,False
4,9,55,13.75,13.88,161,258.92,161,276.23,161,708.17,3,0,-4.589467,137.441633,-4501.04,-2.42,-3.62,111.52,False
5,9,58,14.50,14.62,300,253.85,300,273.28,300,701.24,3,0,-4.589467,137.441633,-4501.04,-2.42,-3.62,111.52,False
6,9,61,15.25,15.38,548,252.96,548,267.09,548,695.50,3,0,-4.589467,137.441633,-4501.04,-2.42,-3.62,111.52,False
7,9,62,15.50,15.62,790,252.16,790,264.67,790,694.19,3,0,-4.589467,137.441633,-4501.04,-2.42,-3.62,111.52,False
8,9,63,15.75,15.88,462,251.90,462,262.38,462,693.00,3,0,-4.589467,137.441633,-4501.04,-2.42,-3.62,111.52,False
9,9,66,16.50,16.62,300,246.58,300,252.44,300,690.95,3,0,-4.589467,137.441633,-4501.04,-2.42,-3.62,111.52,False


In [9]:
out_df.columns.to_list()

['sol',
 'bin',
 'ltst_bin_start_hr',
 'ltst_bin_center_hr',
 'n_air_temp_k',
 'air_temp_k',
 'n_ground_temp_k',
 'ground_temp_k',
 'n_pressure_pa',
 'pressure_pa',
 'rover_site',
 'rover_drive',
 'latitude_deg',
 'longitude_deg_e',
 'elevation_m',
 'roll_deg',
 'pitch_deg',
 'yaw_deg',
 'loc_row_this_sol']